In [1]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

# 1. Setup our realistic parameters
hubs = ['Hyderabad', 'Mumbai', 'Delhi', 'Chennai', 'Bangalore']
carriers = ['BlueDart', 'FedEx', 'Delhivery', 'DHL', 'Shadowfax']
products = ['Insulin Glargine', 'COVID-19 mRNA Vaccine', 'Amoxicillin', 'Humira (Adalimumab)']

# 2. Generate Base Data
np.random.seed(42)
num_records = 5000

data = {
    'Shipment_ID': [f"SHP-{random.randint(10000, 99999)}" for _ in range(num_records)],
    'Product': np.random.choice(products, num_records),
    'Origin': np.random.choice(hubs, num_records),
    'Destination': np.random.choice(hubs, num_records),
    'Carrier': np.random.choice(carriers, num_records),
    'Spoiled_Flag': np.random.choice([0, 1], num_records, p=[0.94, 0.06]) # ~6% spoilage rate
}

df = pd.DataFrame(data)

# 3. Add Dates (and intentionally break some formats)
start_date = datetime(2025, 1, 1)
departure_dates = [start_date + timedelta(days=random.randint(0, 365)) for _ in range(num_records)]
arrival_dates = [d + timedelta(hours=random.randint(12, 72)) for d in departure_dates]

# Intentionally create "dirty" date strings
df['Departure_Time'] = [d.strftime('%Y-%m-%d %H:%M:%S') if random.random() > 0.2 else d.strftime('%d/%m/%Y %H:%M') for d in departure_dates]
df['Arrival_Time'] = [d.strftime('%Y-%m-%d %H:%M:%S') for d in arrival_dates]

# 4. Inject "Enterprise Dirt" (Nulls, Typos, Whitespace)
# Make 5% of Carriers NULL
df.loc[df.sample(frac=0.05).index, 'Carrier'] = np.nan

# Add trailing spaces and lowercase typos to Destinations
df['Destination'] = df['Destination'].apply(lambda x: x + " " if random.random() > 0.8 else x)
df['Destination'] = df['Destination'].apply(lambda x: x.lower() if random.random() > 0.9 else x)

# 5. Export
df.to_csv('dirty_pharma_shipments.csv', index=False)
print("Dataset generated successfully: 'dirty_pharma_shipments.csv'")
print("Rows:", len(df))

Dataset generated successfully: 'dirty_pharma_shipments.csv'
Rows: 5000


In [2]:
import requests
import pandas as pd
import time

# 1. Define our Logistics Hubs and their coordinates (Lat, Lon)
hubs = {
    'Hyderabad': (17.3850, 78.4867),
    'Mumbai': (19.0760, 72.8777),
    'Delhi': (28.7041, 77.1025),
    'Chennai': (13.0827, 80.2707),
    'Bangalore': (12.9716, 77.5946)
}

# 2. Setup Open-Meteo API parameters (Historical Data)
start_date = '2025-01-01'
end_date = '2025-06-30' # First half of the year
base_url = "https://archive-api.open-meteo.com/v1/archive"

all_weather_data = []

print("Fetching historical weather data. This will take a few seconds...")

# 3. Loop through each hub and pull data
for city, coords in hubs.items():
    params = {
        "latitude": coords[0],
        "longitude": coords[1],
        "start_date": start_date,
        "end_date": end_date,
        "daily": "temperature_2m_max",
        "timezone": "Asia/Kolkata"
    }
    
    response = requests.get(base_url, params=params)
    
    if response.status_code == 200:
        data = response.json()
        
        # Open-Meteo returns data in lists. Let's zip them into rows.
        dates = data['daily']['time']
        max_temps = data['daily']['temperature_2m_max']
        
        for date, temp in zip(dates, max_temps):
            all_weather_data.append({
                'Hub_City': city,
                'Date': date,
                'Max_Temp_C': temp
            })
        print(f"✅ Downloaded data for {city}")
    else:
        print(f"❌ Failed to get data for {city}. Status: {response.status_code}")
    
    # Be polite to the API, wait half a second between requests
    time.sleep(0.5)

# 4. Save to CSV
weather_df = pd.DataFrame(all_weather_data)
weather_df.to_csv('weather_data.csv', index=False)
print("\nWeather data successfully saved to 'weather_data.csv'")

Fetching historical weather data. This will take a few seconds...
✅ Downloaded data for Hyderabad
✅ Downloaded data for Mumbai
✅ Downloaded data for Delhi
✅ Downloaded data for Chennai
✅ Downloaded data for Bangalore

Weather data successfully saved to 'weather_data.csv'
